In [3]:
%pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import os

def consolidar_base_mestre(diretorio_dados, data_inicio="2010-01-01", data_fim="2025-12-31"):
    print("=== INÍCIO DA CONSOLIDAÇÃO DA BASE MESTRE (MACRO + PREÇOS) ===")
    
    # 1. Carregamento dos arquivos
    # Nota: Ajustei os nomes conforme os snippets que enviaste
    df_precos = pd.read_csv(os.path.join(diretorio_dados, "precos_limpos_finais.csv"), index_col='Data', parse_dates=True)
    
    # CDI: coluna 'data' e 'valor'
    df_cdi = pd.read_excel(os.path.join(diretorio_dados, "CDI_2010_2026.xlsx"))
    df_cdi = df_cdi.rename(columns={'Date': 'Data', 'valor': 'CDI'}).set_index('Data')
    df_cdi.index = pd.to_datetime(df_cdi.index)
    
    # IBOV: coluna 'Date' e 'IBOV'
    df_ibov = pd.read_excel(os.path.join(diretorio_dados, "IBOV_2010_2026.xlsx"))
    df_ibov = df_ibov.rename(columns={'Date': 'Data'}).set_index('Data')
    df_ibov.index = pd.to_datetime(df_ibov.index)
    
    # SELIC: coluna 'Date' e 'SELIC'
    df_selic = pd.read_excel(os.path.join(diretorio_dados, "SELIC_2010_2026.xlsx"))
    df_selic = df_selic.rename(columns={'Date': 'Data'}).set_index('Data')
    df_selic.index = pd.to_datetime(df_selic.index)

    # 2. Fusão das Bases (Inner Join)
    # Concatenamos todas as colunas. O 'inner' garante que só ficam as datas comuns a todos.
    print("2. A realizar o alinhamento cronológico (Merging)...")
    df_mestre = pd.concat([df_precos, df_ibov, df_cdi, df_selic], axis=1, join='inner')
    #df_mestre = pd.concat([df_precos, df_ibov], axis=1, join='inner')
    
    # 3. Recorte Temporal Restritivo (Evitar Look-ahead Bias)
    print(f"3. A aplicar filtro temporal: {data_inicio} até {data_fim}")
    df_mestre = df_mestre.loc[data_inicio:data_fim]
    
    # 4. Exportação
    caminho_saida = os.path.join(diretorio_dados, "base_mestre_consolidada_IBOV_CDI_SELIC.csv")
    df_mestre.to_csv(caminho_saida)
    
    print("\n=== RESUMO DA CONSOLIDAÇÃO ===")
    print(f"Período Final: {df_mestre.index.min().date()} a {df_mestre.index.max().date()}")
    print(f"Total de Pregões Alinhados: {len(df_mestre)}")
    print(f"Colunas Macro Integradas: IBOV")
    print(f"Arquivo salvo em: {caminho_saida}")
    print("===============================================================")
    
    return df_mestre

# Execução
pasta_dados = r"C:\VSCodeWorkspace\TCC_Escrito\data"
base_mestre = consolidar_base_mestre(pasta_dados)

=== INÍCIO DA CONSOLIDAÇÃO DA BASE MESTRE (MACRO + PREÇOS) ===
2. A realizar o alinhamento cronológico (Merging)...
3. A aplicar filtro temporal: 2010-01-01 até 2025-12-31

=== RESUMO DA CONSOLIDAÇÃO ===
Período Final: 2010-01-04 a 2025-12-30
Total de Pregões Alinhados: 3964
Colunas Macro Integradas: IBOV
Arquivo salvo em: C:\VSCodeWorkspace\TCC_Escrito\data\base_mestre_consolidada_IBOV_CDI_SELIC.csv
